# Session 1: a triangle is rigid, a rectangle is not
**A. Real-world problem.** Look under a railway bridge or at a crane arm: you see
triangles made of straight bars. Why triangles, and not rectangles? And when a heavy
lorry stands in the middle, which bars are being *pulled* and which are being *squashed*?

**Plan.** (1) show that four bars joined by pins can lean over but three cannot,
(2) count the bars a rigid frame needs, (3) find the forces in the simplest bridge
truss, (4) draw it with tension in blue and compression in red.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: a rectangle of pinned bars can lean, a triangle cannot
Take a 4 m by 3 m rectangle of four bars joined by pins. Push its top sideways: the bar
lengths stay the same but the corner angle changes (the frame "racks"). Now add a
diagonal bar. The diagonal of a parallelogram with sides a and b and corner angle phi has
length d = sqrt(a^2 + b^2 - 2 a b cos(phi)). A diagonal of fixed length therefore allows
only one angle: the frame can no longer lean.

In [ ]:
def diagonal_length(a, b, phi_deg):
    """Length of the diagonal opposite the corner angle phi of a pinned parallelogram."""
    phi = np.radians(phi_deg)
    return float(np.sqrt(a * a + b * b - 2 * a * b * np.cos(phi)))


def corner_angle_for_diagonal(a, b, d):
    """The single corner angle (degrees) that a fixed diagonal d allows (law of cosines)."""
    c = (a * a + b * b - d * d) / (2 * a * b)
    return float(np.degrees(np.arccos(np.clip(c, -1.0, 1.0))))

## Step 2: how many bars make a rigid frame?
A frame with j pins in a plane needs at least 2 j - 3 bars to be rigid. A triangle:
j = 3, so 3 bars. A rectangle: j = 4, so 5 bars, but it has only 4.

In [ ]:
def bars_needed(joints):
    """Smallest number of bars that can make a plane pin-jointed frame rigid."""
    return 2 * joints - 3


def is_rigid_by_count(joints, bars):
    """Necessary condition only: enough bars. (Bars must also be placed sensibly.)"""
    return bars >= bars_needed(joints)

## Step 3: the simplest bridge truss
Two equal sloping bars meet at the middle (the apex, where the load P hangs). A bottom
bar joins the two feet. Span L, height h. The supports hold up P/2 each. At the apex,
the two sloping bars share the load: 2 F sin(theta) = P, where theta is the slope angle.
The sloping bars are squashed (compression), the bottom bar is stretched (tension).
We write tension as positive and compression as negative.

In [ ]:
def slope_angle(span, height):
    """Angle (radians) of the sloping bars above the horizontal."""
    return float(np.arctan2(height, span / 2.0))


def triangle_truss_forces(load, span, height):
    """Return (F_top, F_bottom) in the units of load. F_top < 0 is compression, F_bottom > 0 tension."""
    th = slope_angle(span, height)
    f_top = -load / (2.0 * np.sin(th))
    f_bottom = load / (2.0 * np.tan(th))
    return float(f_top), float(f_bottom)


def force_colour(force, tol=1e-9):
    """blue for tension, red for compression, grey for a zero-force bar."""
    if force > tol:
        return "tab:blue"
    if force < -tol:
        return "tab:red"
    return "0.6"

## Step 4: see it
Left: the rectangle, the leaning rectangle and the braced triangle frame. Right: the
bridge with a load in the middle; line width shows the size of the force.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    a, b = 4.0, 3.0
    print("rectangle 4 x 3: diagonal at 90 degrees =", round(diagonal_length(a, b, 90), 3), "m")
    print("leaning to 60 degrees: diagonal =", round(diagonal_length(a, b, 60), 3), "m")
    print("a fixed 5 m diagonal allows only", round(corner_angle_for_diagonal(a, b, 5.0), 1), "degrees")
    print("bars needed: triangle", bars_needed(3), "rectangle", bars_needed(4))

    fig, axs = plt.subplots(1, 3, figsize=(9, 3))
    for ax, phi, title, diag in ((axs[0], 90, "rectangle: 4 bars", False),
                                 (axs[1], 60, "leaning: same 4 bars", False),
                                 (axs[2], 90, "with a diagonal: rigid", True)):
        p = np.radians(phi)
        pts = np.array([[0, 0], [a, 0], [a + b * np.cos(p), b * np.sin(p)], [b * np.cos(p), b * np.sin(p)]])
        ax.fill(pts[:, 0], pts[:, 1], alpha=0.2)
        ax.plot(*np.vstack([pts, pts[0]]).T, "o-", color="k")
        if diag:
            ax.plot([pts[0, 0], pts[2, 0]], [pts[0, 1], pts[2, 1]], "-", color="tab:red", lw=3)
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=9)
        ax.set_xlabel("x (m)")
        ax.set_xlim(-1, 9)
        ax.set_ylim(-0.5, 3.5)
    axs[0].set_ylabel("y (m)")
    fig.tight_layout()
    fig.savefig(OUT / "rigid_vs_rectangle.png", dpi=120)

    span, load = 6.0, 10.0
    fig2, (axb, axf) = plt.subplots(1, 2, figsize=(10, 3.8))
    height = 3.0
    ft, fb = triangle_truss_forces(load, span, height)
    print(f"span 6 m, height 3 m, load 10 kN: top bars {ft:.2f} kN, bottom bar {fb:.2f} kN")
    nodes = np.array([[0, 0], [span, 0], [span / 2, height]])
    for (i, j), f in (((0, 2), ft), ((2, 1), ft), ((0, 1), fb)):
        axb.plot(nodes[[i, j], 0], nodes[[i, j], 1], color=force_colour(f), lw=1 + 5 * abs(f) / 10)
    axb.annotate("", xy=(span / 2, height), xytext=(span / 2, height + 1.2),
                 arrowprops=dict(arrowstyle="->", lw=2))
    axb.text(span / 2 + 0.2, height + 0.7, "10 kN")
    axb.text(0.1, -0.7, "blue: tension +5.00 kN, red: compression -7.07 kN", fontsize=8)
    axb.set_aspect("equal")
    axb.set_xlabel("x (m)")
    axb.set_ylabel("y (m)")
    axb.set_ylim(-1.0, 5.2)
    axb.set_title("Simple bridge truss, load in the middle")
    hs = np.linspace(0.5, 4.0, 60)
    res = np.array([triangle_truss_forces(load, span, h) for h in hs])
    axf.plot(hs, res[:, 0], color="tab:red", label="sloping bars (compression)")
    axf.plot(hs, res[:, 1], color="tab:blue", label="bottom bar (tension)")
    axf.axhline(0, color="gray", lw=0.8)
    axf.set_xlabel("truss height h (m)")
    axf.set_ylabel("bar force (kN, tension positive)")
    axf.set_title("A taller truss carries the load with smaller forces")
    axf.grid(alpha=0.3)
    axf.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "simple_truss.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the height in `lesson.py` to 2 m and 4 m and print both forces. Predict the order of the sizes before you run it.
2. **Stretch.** Write `parallelogram_area(a, b, phi_deg)` that returns a * b * sin(phi) and show that a rectangle leaning from 90 to 60 degrees loses area although the bars keep their lengths.
3. **Explain.** Explain, with a sketch, why the bottom bar of the simple bridge truss must be in tension when the supports are a pin and a roller.

## Assessment
- **A. Conceptual.** Why does adding one diagonal make a rectangle of pinned bars rigid, and why does the rectangle without it fail? Use the numbers of the 4 m by 3 m rectangle.
- **B. Numerical.** A truss of span 6 m and height 1.5 m carries 12 kN at its apex. Find the force in the sloping bars and in the bottom bar, with signs.
- **C. Graph and simulation interpretation.** On the force against height graph (10 kN load), what happens to the forces as the truss is made taller, from 0.5 m to 4 m? Give the forces at both ends.
- **D. Design and optimisation challenge.** Design: choose the smallest height (to the nearest 0.25 m) so that the bottom bar's tension is at most 4 kN under a 10 kN load on a 6 m span. What compression does each sloping bar then carry, and what do you give up?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("a 4 x 3 rectangle at 90 degrees has a 5.0 m diagonal",
                   abs(L.diagonal_length(4.0, 3.0, 90.0) - 5.0) < 1e-9))
    checks.append(("leaning the rectangle to 60 degrees shortens the diagonal to 3.606 m",
                   abs(L.diagonal_length(4.0, 3.0, 60.0) - 3.606) < 1e-3))
    checks.append(("a fixed 5 m diagonal allows only the 90 degree corner",
                   abs(L.corner_angle_for_diagonal(4.0, 3.0, 5.0) - 90.0) < 1e-6))
    checks.append(("a triangle needs 3 bars and a rectangle needs 5",
                   L.bars_needed(3) == 3 and L.bars_needed(4) == 5))
    checks.append(("a four-bar rectangle is not rigid, a triangle is",
                   (not L.is_rigid_by_count(4, 4)) and L.is_rigid_by_count(3, 3)))
    ft, fb = L.triangle_truss_forces(10.0, 6.0, 3.0)
    checks.append(("worked example: 10 kN, span 6 m, height 3 m gives -7.07 kN (top) and +5.00 kN (bottom)",
                   abs(ft + 7.07) < 0.005 and abs(fb - 5.00) < 0.005))
    ft2, fb2 = L.triangle_truss_forces(10.0, 6.0, 2.0)
    checks.append(("height 2 m gives -9.01 kN and +7.50 kN (a flatter truss pulls and pushes harder)",
                   abs(ft2 + 9.01) < 0.005 and abs(fb2 - 7.50) < 0.005))
    # hand calculation: equilibrium of the apex pin, sum Fy = 0 -> -2 F_top sin(theta) = P ... F_top is negative
    th = L.slope_angle(6.0, 3.0)
    checks.append(("apex pin: sum of vertical forces is zero (2 |F| sin(theta) = P)",
                   abs(2 * abs(ft) * np.sin(th) - 10.0) < 1e-9))
    # right foot pin: horizontal parts balance (bottom bar pulls inwards, sloping bar pushes outwards)
    checks.append(("foot pin: sum of horizontal forces is zero (|F_top| cos(theta) = F_bottom)",
                   abs(abs(ft) * np.cos(th) - fb) < 1e-9))
    checks.append(("force is proportional to load (doubling the load doubles both forces)",
                   np.allclose(L.triangle_truss_forces(20.0, 6.0, 3.0), [2 * ft, 2 * fb])))
    f12 = L.triangle_truss_forces(12.0, 6.0, 1.5)
    checks.append(("12 kN, height 1.5 m: top bars -13.42 kN and bottom bar +12.00 kN",
                   abs(f12[0] + 13.416) < 0.001 and abs(f12[1] - 12.0) < 1e-9))
    f375 = L.triangle_truss_forces(10.0, 6.0, 3.75)
    checks.append(("design: height 3.75 m keeps the bottom tension at 4.00 kN (top bars -6.40 kN)",
                   abs(f375[1] - 4.0) < 1e-9 and abs(f375[0] + 6.403) < 0.001))
    checks.append(("height 3 m means a 45 degree slope, height 2 m means 33.7 degrees",
                   abs(np.degrees(L.slope_angle(6.0, 3.0)) - 45.0) < 1e-9 and abs(np.degrees(L.slope_angle(6.0, 2.0)) - 33.69) < 0.005
                   and abs(np.degrees(L.slope_angle(6.0, 3.75)) - 51.34) < 0.005))
    checks.append(("12 kN, height 1.5 m: the vertical part of a sloping bar's force is 6.0 kN, half the load",
                   abs(abs(f12[0]) * np.sin(L.slope_angle(6.0, 1.5)) - 6.0) < 1e-9))
    f05 = L.triangle_truss_forces(10.0, 6.0, 0.5)
    checks.append(("height 0.5 m gives 30.41 kN compression and 30.00 kN tension; height 4 m gives 6.25 and 3.75 kN",
                   abs(f05[0] + 30.414) < 0.001 and abs(f05[1] - 30.0) < 1e-9
                   and np.allclose(L.triangle_truss_forces(10.0, 6.0, 4.0), [-6.25, 3.75], atol=1e-9)))
    checks.append(("colours: tension blue, compression red",
                   L.force_colour(5.0) == "tab:blue" and L.force_colour(-5.0) == "tab:red"))
    # limit: a very flat truss gives enormous forces
    f_flat = L.triangle_truss_forces(10.0, 6.0, 0.01)
    checks.append(("limit: an almost flat truss (h = 0.01 m) has forces over 1000 kN", abs(f_flat[0]) > 1000))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
